In [1]:
from google.colab import drive

drive.mount("/content/gdrive")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


Install GIS libraries

In [2]:
!pip install -q osmnx folium geopandas shapely

Import libraries

In [3]:
import osmnx as ox
import geopandas as gpd
import folium
import pandas as pd

from shapely.geometry import box, Point
from folium.plugins import MousePosition

print("CrisisWeave GIS initialized successfully.")

CrisisWeave GIS initialized successfully.


GIS-03 — Define our small study area

In [4]:
# ==========================================
# CrisisWeave Study Area
# ==========================================

CENTER_LAT = 9.9283
CENTER_LON = 76.2683

# Approximately 1 km × 1 km
LAT_OFFSET = 0.0045
LON_OFFSET = 0.0045

NORTH = CENTER_LAT + LAT_OFFSET
SOUTH = CENTER_LAT - LAT_OFFSET

EAST = CENTER_LON + LON_OFFSET
WEST = CENTER_LON - LON_OFFSET

print("CrisisWeave Study Area")
print("----------------------")
print(f"North : {NORTH:.6f}")
print(f"South : {SOUTH:.6f}")
print(f"East  : {EAST:.6f}")
print(f"West  : {WEST:.6f}")

CrisisWeave Study Area
----------------------
North : 9.932800
South : 9.923800
East  : 76.272800
West  : 76.263800


GIS-04 — Create the clean interactive map

In [5]:
# ==========================================
# Clean CrisisWeave Interactive Map
# ==========================================

crisis_map = folium.Map(
    location=[CENTER_LAT, CENTER_LON],
    zoom_start=16,
    min_zoom=16,
    max_zoom=18,
    tiles="OpenStreetMap",
    control_scale=True
)

# Keep the user inside the selected study area
crisis_map.fit_bounds([
    [SOUTH, WEST],
    [NORTH, EAST]
])

crisis_map.options["maxBounds"] = [
    [SOUTH, WEST],
    [NORTH, EAST]
]

crisis_map.options["maxBoundsViscosity"] = 1.0

crisis_map

GIS-05 — Add mouse coordinates

In [6]:
# ==========================================
# Mouse Latitude / Longitude
# ==========================================

MousePosition(
    position="bottomright",
    separator=" | ",
    prefix="Coordinates:",
    num_digits=6
).add_to(crisis_map)

crisis_map

GIS-06 — Get REAL schools and hospitals

In [7]:
# ==========================================
# Get REAL GIS Features
# ==========================================

bbox = (WEST, SOUTH, EAST, NORTH)

# Schools
schools = ox.features_from_bbox(
    bbox=bbox,
    tags={"amenity": "school"}
).reset_index()

# Hospitals
hospitals = ox.features_from_bbox(
    bbox=bbox,
    tags={"amenity": "hospital"}
).reset_index()

print("Real OSM features found")
print("-----------------------")
print("Schools   :", len(schools))
print("Hospitals :", len(hospitals))

Real OSM features found
-----------------------
Schools   : 1
Hospitals : 1


GIS-07 — Get REAL bridges

In [8]:
# ==========================================
# Get REAL Bridge Geometry
# ==========================================

bridges = ox.features_from_bbox(
    bbox=bbox,
    tags={"bridge": True}
).reset_index()

# Keep line geometry only
bridges = bridges[
    bridges.geometry.geom_type.isin(
        ["LineString", "MultiLineString"]
    )
].copy()

print("Real bridges found:", len(bridges))

if len(bridges) > 0:

    display_columns = [
        c for c in
        ["name", "highway", "bridge"]
        if c in bridges.columns
    ]

    print(bridges[display_columns].head(10))

Real bridges found: 2
                      name      highway bridge
0  Gateway of Kochi Bridge      primary    yes
1                      NaN  residential    yes


GIS-08 — Add only important GIS features

In [9]:
# ==========================================
# Add GIS Feature Markers
# ==========================================

# ------------------------------------------
# Schools
# ------------------------------------------

for _, row in schools.iterrows():

    geom = row.geometry

    if geom.geom_type == "Point":
        point = geom
    else:
        point = geom.centroid

    name = row.get("name", "School")

    if pd.isna(name):
        name = "School"

    folium.Marker(
        location=[point.y, point.x],
        tooltip=f"🏫 {name}",
        popup=f"""
        <b>School</b><br>
        Name: {name}<br>
        Latitude: {point.y:.6f}<br>
        Longitude: {point.x:.6f}
        """,
        icon=folium.Icon(
            icon="graduation-cap",
            prefix="fa"
        )
    ).add_to(crisis_map)


# ------------------------------------------
# Hospitals
# ------------------------------------------

for _, row in hospitals.iterrows():

    geom = row.geometry

    if geom.geom_type == "Point":
        point = geom
    else:
        point = geom.centroid

    name = row.get("name", "Hospital")

    if pd.isna(name):
        name = "Hospital"

    folium.Marker(
        location=[point.y, point.x],
        tooltip=f"🏥 {name}",
        popup=f"""
        <b>Hospital</b><br>
        Name: {name}<br>
        Latitude: {point.y:.6f}<br>
        Longitude: {point.x:.6f}
        """,
        icon=folium.Icon(
            icon="plus",
            prefix="fa"
        )
    ).add_to(crisis_map)


# ------------------------------------------
# Bridges
# ------------------------------------------

for _, row in bridges.iterrows():

    geom = row.geometry

    # Highlight actual bridge geometry
    if geom.geom_type == "LineString":

        coords = [
            [lat, lon]
            for lon, lat in geom.coords
        ]

        folium.PolyLine(
            locations=coords,
            weight=6,
            tooltip=f"🌉 {row.get('name', 'Bridge')}"
        ).add_to(crisis_map)

    elif geom.geom_type == "MultiLineString":

        for line in geom.geoms:

            coords = [
                [lat, lon]
                for lon, lat in line.coords
            ]

            folium.PolyLine(
                locations=coords,
                weight=6,
                tooltip=f"🌉 {row.get('name', 'Bridge')}"
            ).add_to(crisis_map)

    # Put one marker at bridge center
    point = geom.representative_point()

    name = row.get("name", "Bridge")

    if pd.isna(name):
        name = "Bridge"

    folium.Marker(
        location=[point.y, point.x],
        tooltip=f"🌉 {name}",
        popup=f"""
        <b>Bridge</b><br>
        Name: {name}<br>
        Latitude: {point.y:.6f}<br>
        Longitude: {point.x:.6f}
        """,
        icon=folium.Icon(
            icon="road",
            prefix="fa"
        )
    ).add_to(crisis_map)


crisis_map

GIS-09 — ONE incident pin only

In [10]:
# ==========================================
# ONE INCIDENT PIN ONLY
# ==========================================

from branca.element import MacroElement
from jinja2 import Template


class SingleIncidentMarker(MacroElement):

    def __init__(self):

        super().__init__()

        self._name = "SingleIncidentMarker"

        self._template = Template("""
        {% macro script(this, kwargs) %}

        var incidentMarker = null;

        {{this._parent.get_name()}}.on('click', function(e) {

            var lat = e.latlng.lat;
            var lng = e.latlng.lng;

            // Remove previous incident marker
            if (incidentMarker !== null) {
                {{this._parent.get_name()}}
                    .removeLayer(incidentMarker);
            }

            // Create ONE incident marker
            incidentMarker = L.marker(
                [lat, lng]
            ).addTo(
                {{this._parent.get_name()}}
            );

            incidentMarker.bindPopup(
                "<b>📍 CrisisWeave Incident</b><br>" +
                "Latitude: " +
                lat.toFixed(6) +
                "<br>" +
                "Longitude: " +
                lng.toFixed(6)
            ).openPopup();

        });

        {% endmacro %}
        """)


crisis_map.add_child(
    SingleIncidentMarker()
)

crisis_map

GIS-10 — Incident Location Record

In [11]:
# ==========================================
# GIS-10: INCIDENT LOCATION RECORD
# ==========================================

incident_location = {
    "latitude": None,
    "longitude": None,
    "location_source": None
}

print("Incident location record initialized.")
print(incident_location)

Incident location record initialized.
{'latitude': None, 'longitude': None, 'location_source': None}


GIS-11 — Test with a selected location

In [12]:
# ==========================================
# GIS-11: SET INCIDENT LOCATION FOR TEST
# ==========================================

incident_latitude = 9.929117
incident_longitude = 76.270436

incident_location = {
    "latitude": incident_latitude,
    "longitude": incident_longitude,
    "location_source": "user_map_pin"
}

print("Selected Incident Location")
print("--------------------------")
print(f"Latitude : {incident_latitude:.6f}")
print(f"Longitude: {incident_longitude:.6f}")
print(f"Source   : {incident_location['location_source']}")

Selected Incident Location
--------------------------
Latitude : 9.929117
Longitude: 76.270436
Source   : user_map_pin


In [14]:
# ==========================================
# GIS DATA RECOVERY
# Run this BEFORE GIS-12
# ==========================================

import osmnx as ox
import geopandas as gpd
import pandas as pd

from shapely.geometry import Point

# ------------------------------------------
# Study area
# ------------------------------------------

CENTER_LAT = 9.9283
CENTER_LON = 76.2683

LAT_OFFSET = 0.0045
LON_OFFSET = 0.0045

NORTH = CENTER_LAT + LAT_OFFSET
SOUTH = CENTER_LAT - LAT_OFFSET
EAST = CENTER_LON + LON_OFFSET
WEST = CENTER_LON - LON_OFFSET

bbox = (WEST, SOUTH, EAST, NORTH)

print("Study area ready.")


# ==========================================
# REAL ROADS
# ==========================================

roads = ox.features_from_bbox(
    bbox=bbox,
    tags={"highway": True}
).reset_index()

roads = roads[
    roads.geometry.geom_type.isin(
        ["LineString", "MultiLineString"]
    )
].copy()


# ==========================================
# REAL SCHOOLS
# ==========================================

schools = ox.features_from_bbox(
    bbox=bbox,
    tags={"amenity": "school"}
).reset_index()


# ==========================================
# REAL HOSPITALS
# ==========================================

hospitals = ox.features_from_bbox(
    bbox=bbox,
    tags={"amenity": "hospital"}
).reset_index()


# ==========================================
# REAL BRIDGES
# ==========================================

bridges = ox.features_from_bbox(
    bbox=bbox,
    tags={"bridge": True}
).reset_index()

bridges = bridges[
    bridges.geometry.geom_type.isin(
        ["LineString", "MultiLineString"]
    )
].copy()


# ==========================================
# SUMMARY
# ==========================================

print()
print("REAL OSM GIS DATA")
print("-----------------")
print("Roads     :", len(roads))
print("Schools   :", len(schools))
print("Hospitals :", len(hospitals))
print("Bridges   :", len(bridges))

Study area ready.

REAL OSM GIS DATA
-----------------
Roads     : 100
Schools   : 1
Hospitals : 1
Bridges   : 2


GIS-12 — Calculate distance correctly

In [16]:
# ==========================================
# GIS-12: PROJECT GIS DATA FOR DISTANCE
# ==========================================

METRIC_CRS = "EPSG:32643"

incident_point = gpd.GeoDataFrame(
    {
        "type": ["incident"]
    },
    geometry=[
        Point(
            incident_longitude,
            incident_latitude
        )
    ],
    crs="EPSG:4326"
)

incident_metric = incident_point.to_crs(
    METRIC_CRS
)

roads_metric = roads.to_crs(METRIC_CRS)
schools_metric = schools.to_crs(METRIC_CRS)
hospitals_metric = hospitals.to_crs(METRIC_CRS)
bridges_metric = bridges.to_crs(METRIC_CRS)

print("GIS layers projected to metric coordinates.")

GIS layers projected to metric coordinates.


GIS-13 — Find nearest features

In [17]:
# ==========================================
# GIS-13: NEAREST FEATURE ANALYSIS
# ==========================================

def nearest_feature(
    incident_metric,
    feature_gdf,
    feature_type
):

    if feature_gdf is None or len(feature_gdf) == 0:
        return None

    distances = feature_gdf.geometry.distance(
        incident_metric.geometry.iloc[0]
    )

    nearest_index = distances.idxmin()

    feature = feature_gdf.loc[nearest_index]

    distance = float(
        distances.loc[nearest_index]
    )

    name = feature.get(
        "name",
        feature_type
    )

    if pd.isna(name):
        name = feature_type

    return {
        "feature_type": feature_type,
        "name": str(name),
        "distance_m": round(distance, 2)
    }


spatial_context = []

for feature_gdf, feature_type in [
    (roads_metric, "Road"),
    (schools_metric, "School"),
    (hospitals_metric, "Hospital"),
    (bridges_metric, "Bridge")
]:

    result = nearest_feature(
        incident_metric,
        feature_gdf,
        feature_type
    )

    if result:
        spatial_context.append(result)


spatial_context

[{'feature_type': 'Road', 'name': 'Syrian Church Road', 'distance_m': 17.2},
 {'feature_type': 'School',
  'name': 'Kendriya Vidyalaya-Mundamveli Kochi',
  'distance_m': 358.26},
 {'feature_type': 'Hospital', 'name': 'AMJ Hospital', 'distance_m': 38.14},
 {'feature_type': 'Bridge',
  'name': 'Gateway of Kochi Bridge',
  'distance_m': 381.73}]

GIS-14 — Display spatial context

In [18]:
# ==========================================
# GIS-14: INCIDENT SPATIAL CONTEXT
# ==========================================

spatial_context_df = pd.DataFrame(
    spatial_context
)

spatial_context_df

,feature_type,name,distance_m
0,Road,Syrian Church Road,17.20
1,School,Kendriya Vidyalaya-Mundamveli Kochi,358.26
2,Hospital,AMJ Hospital,38.14
3,Bridge,Gateway of Kochi Bridge,381.73


GIS-15 — Add this to our CrisisWeave record

In [19]:
# ==========================================
# GIS-15: CRISISWEAVE INCIDENT RECORD
# ==========================================

crisis_incident = {
    "incident_id": "CW-001",

    "location": {
        "latitude": incident_latitude,
        "longitude": incident_longitude,
        "source": "user_map_pin"
    },

    "gis_context": spatial_context
}

print("CrisisWeave Incident Record")
print("===========================")

print(
    "Incident ID:",
    crisis_incident["incident_id"]
)

print(
    "Latitude:",
    crisis_incident["location"]["latitude"]
)

print(
    "Longitude:",
    crisis_incident["location"]["longitude"]
)

print("\nGIS Context:")

for item in crisis_incident["gis_context"]:
    print(
        f"- {item['feature_type']}: "
        f"{item['name']} "
        f"({item['distance_m']} m)"
    )

CrisisWeave Incident Record
Incident ID: CW-001
Latitude: 9.929117
Longitude: 76.270436

GIS Context:
- Road: Syrian Church Road (17.2 m)
- School: Kendriya Vidyalaya-Mundamveli Kochi (358.26 m)
- Hospital: AMJ Hospital (38.14 m)
- Bridge: Gateway of Kochi Bridge (381.73 m)


GIS-16 — Waterbody Spatial Relationship.

In [20]:
# ==========================================
# GIS-16: WATERBODY SPATIAL RELATIONSHIP
# ==========================================

# Get real OSM water features
water = ox.features_from_bbox(
    bbox=bbox,
    tags={
        "natural": "water",
        "waterway": True
    }
).reset_index()

print("Water features found:", len(water))

Water features found: 17


GIS-17 — Project water features

In [21]:
# ==========================================
# GIS-17: PROJECT WATER FEATURES
# ==========================================

water_metric = water.to_crs(METRIC_CRS)

print("Water layer projected successfully.")

Water layer projected successfully.


GIS-18 — Calculate relationship with incident

In [22]:
# ==========================================
# GIS-18: INCIDENT-WATER RELATIONSHIP
# ==========================================

incident_geom = incident_metric.geometry.iloc[0]

water_results = []

for idx, row in water_metric.iterrows():

    geom = row.geometry

    # Skip invalid/empty geometry
    if geom is None or geom.is_empty:
        continue

    distance = incident_geom.distance(geom)

    # Check whether incident is inside water
    inside = geom.contains(incident_geom)

    # Classify spatial relationship
    if inside:
        relationship = "Inside waterbody"

    elif distance <= 50:
        relationship = "Adjacent to waterbody"

    elif distance <= 200:
        relationship = "Near waterbody"

    else:
        relationship = "Far from waterbody"

    name = row.get("name", "Waterbody")

    if pd.isna(name):
        name = "Waterbody"

    water_results.append({
        "waterbody": str(name),
        "distance_m": round(float(distance), 2),
        "relationship": relationship
    })


water_relationship_df = pd.DataFrame(
    water_results
)

# Sort nearest first
if len(water_relationship_df) > 0:
    water_relationship_df = (
        water_relationship_df
        .sort_values("distance_m")
        .reset_index(drop=True)
    )

water_relationship_df.head(10)

,waterbody,distance_m,relationship
0,Waterbody,73.52,Near waterbody
1,Vembanad Lake,73.52,Near waterbody
2,Waterbody,127.55,Near waterbody
3,Waterbody,216.30,Far from waterbody
4,Waterbody,235.93,Far from waterbody
5,Waterbody,241.01,Far from waterbody
6,Waterbody,330.67,Far from waterbody
7,Waterbody,345.38,Far from waterbody
8,Waterbody,372.34,Far from waterbody
9,Waterbody,383.80,Far from waterbody


GIS-19 — Get the closest water feature

In [23]:
# ==========================================
# GIS-19: NEAREST WATER FEATURE
# ==========================================

if len(water_relationship_df) > 0:

    nearest_water = water_relationship_df.iloc[0]

    print("Nearest Water Feature")
    print("---------------------")
    print("Name         :", nearest_water["waterbody"])
    print("Distance     :", nearest_water["distance_m"], "meters")
    print("Relationship :", nearest_water["relationship"])

else:

    nearest_water = None

    print("No water feature found in the study area.")

Nearest Water Feature
---------------------
Name         : Waterbody
Distance     : 73.52 meters
Relationship : Near waterbody


GIS-20: UNIFIED GIS CONTEXT

In [24]:
# ==========================================
# GIS-20: UNIFIED GIS CONTEXT
# ==========================================

def get_nearest_context(
    incident_metric,
    feature_gdf,
    feature_type
):
    """
    Find the nearest geographic feature
    and return its name, coordinates and distance.
    """

    if feature_gdf is None or len(feature_gdf) == 0:
        return {
            "feature_type": feature_type,
            "name": None,
            "latitude": None,
            "longitude": None,
            "distance_m": None
        }

    distances = feature_gdf.geometry.distance(
        incident_metric.geometry.iloc[0]
    )

    nearest_index = distances.idxmin()

    feature = feature_gdf.loc[nearest_index]

    distance_m = float(
        distances.loc[nearest_index]
    )

    # Get representative point
    point = feature.geometry.representative_point()

    name = feature.get(
        "name",
        feature_type
    )

    if pd.isna(name):
        name = feature_type

    return {
        "feature_type": feature_type,
        "name": str(name),
        "latitude": round(float(point.y), 6),
        "longitude": round(float(point.x), 6),
        "distance_m": round(distance_m, 2)
    }


# ==========================================
# Build GIS context
# ==========================================

gis_context = {}

gis_context["nearest_road"] = get_nearest_context(
    incident_metric,
    roads_metric,
    "Road"
)

gis_context["nearest_school"] = get_nearest_context(
    incident_metric,
    schools_metric,
    "School"
)

gis_context["nearest_hospital"] = get_nearest_context(
    incident_metric,
    hospitals_metric,
    "Hospital"
)

gis_context["nearest_bridge"] = get_nearest_context(
    incident_metric,
    bridges_metric,
    "Bridge"
)


# Waterbody comes from our previous analysis
if nearest_water is not None:

    gis_context["nearest_waterbody"] = {
        "feature_type": "Waterbody",
        "name": nearest_water["waterbody"],
        "distance_m": float(
            nearest_water["distance_m"]
        ),
        "relationship": nearest_water["relationship"]
    }

else:

    gis_context["nearest_waterbody"] = {
        "feature_type": "Waterbody",
        "name": None,
        "distance_m": None,
        "relationship": None
    }


print("GIS context created successfully.")

GIS context created successfully.


GIS-21 — Display the GIS context

In [25]:
# ==========================================
# GIS-21: DISPLAY GIS CONTEXT
# ==========================================

for key, value in gis_context.items():

    print(f"\n{key}")
    print("-" * len(key))

    for field, data in value.items():
        print(f"{field}: {data}")


nearest_road
------------
feature_type: Road
name: Syrian Church Road
latitude: 1097835.502269
longitude: 639258.587067
distance_m: 17.2

nearest_school
--------------
feature_type: School
name: Kendriya Vidyalaya-Mundamveli Kochi
latitude: 1098032.308252
longitude: 638927.3767
distance_m: 358.26

nearest_hospital
----------------
feature_type: Hospital
name: AMJ Hospital
latitude: 1097876.28905
longitude: 639258.858736
distance_m: 38.14

nearest_bridge
--------------
feature_type: Bridge
name: Gateway of Kochi Bridge
latitude: 1098348.974526
longitude: 639212.568086
distance_m: 381.73

nearest_waterbody
-----------------
feature_type: Waterbody
name: Waterbody
distance_m: 73.52
relationship: Near waterbody


GIS-22 — Put it into the CrisisWeave incident record

In [26]:
# ==========================================
# GIS-22: CRISISWEAVE INCIDENT RECORD
# ==========================================

crisis_incident = {

    "incident_id": "CW-001",

    "location": {
        "latitude": incident_latitude,
        "longitude": incident_longitude,
        "source": "user_map_pin"
    },

    "gis_context": gis_context
}


print("===================================")
print("CRISISWEAVE INCIDENT RECORD")
print("===================================")

print(
    f"Incident ID : "
    f"{crisis_incident['incident_id']}"
)

print(
    f"Latitude    : "
    f"{crisis_incident['location']['latitude']}"
)

print(
    f"Longitude   : "
    f"{crisis_incident['location']['longitude']}"
)

print("\nGIS CONTEXT")
print("-----------")

for key, value in gis_context.items():

    print(
        f"{key}: "
        f"{value.get('name')} "
        f"({value.get('distance_m')} m)"
    )

CRISISWEAVE INCIDENT RECORD
Incident ID : CW-001
Latitude    : 9.929117
Longitude   : 76.270436

GIS CONTEXT
-----------
nearest_road: Syrian Church Road (17.2 m)
nearest_school: Kendriya Vidyalaya-Mundamveli Kochi (358.26 m)
nearest_hospital: AMJ Hospital (38.14 m)
nearest_bridge: Gateway of Kochi Bridge (381.73 m)
nearest_waterbody: Waterbody (73.52 m)


GIS-23 — Location Verification

In [27]:
# ==========================================
# GIS-23: LOCATION VERIFICATION
# ==========================================

def verify_incident_location(
    gis_context,
    max_water_distance=200,
    max_road_distance=100
):
    """
    Verify whether the selected incident location
    has a plausible geographic relationship with
    the mapped study-area features.

    This is NOT a flood prediction model.
    It is a geographic plausibility check.
    """

    water = gis_context.get(
        "nearest_waterbody",
        {}
    )

    road = gis_context.get(
        "nearest_road",
        {}
    )

    bridge = gis_context.get(
        "nearest_bridge",
        {}
    )

    school = gis_context.get(
        "nearest_school",
        {}
    )

    hospital = gis_context.get(
        "nearest_hospital",
        {}
    )

    # --------------------------------------
    # Water evidence
    # --------------------------------------

    water_distance = water.get(
        "distance_m"
    )

    if water_distance is not None:

        water_relevant = (
            water_distance <= max_water_distance
        )

    else:

        water_relevant = False


    # --------------------------------------
    # Road evidence
    # --------------------------------------

    road_distance = road.get(
        "distance_m"
    )

    if road_distance is not None:

        road_relevant = (
            road_distance <= max_road_distance
        )

    else:

        road_relevant = False


    # --------------------------------------
    # Geographic plausibility
    # --------------------------------------

    verification_score = 0

    if water_relevant:
        verification_score += 1

    if road_relevant:
        verification_score += 1

    if bridge.get("distance_m") is not None:
        verification_score += 1

    if school.get("distance_m") is not None:
        verification_score += 1

    if hospital.get("distance_m") is not None:
        verification_score += 1


    # --------------------------------------
    # Final classification
    # --------------------------------------

    if verification_score >= 3:

        status = "GEOGRAPHICALLY PLAUSIBLE"

    elif verification_score >= 2:

        status = "PARTIALLY PLAUSIBLE"

    else:

        status = "INSUFFICIENT GIS EVIDENCE"


    return {
        "status": status,
        "verification_score": verification_score,
        "water_relevant": water_relevant,
        "road_relevant": road_relevant,
        "water_distance_m": water_distance,
        "road_distance_m": road_distance
    }


location_verification = verify_incident_location(
    gis_context
)

location_verification

{'status': 'GEOGRAPHICALLY PLAUSIBLE',
 'verification_score': 5,
 'water_relevant': True,
 'road_relevant': True,
 'water_distance_m': 73.52,
 'road_distance_m': 17.2}

GIS-24 — Display verification result

In [28]:
# ==========================================
# GIS-24: LOCATION VERIFICATION RESULT
# ==========================================

print("======================================")
print("CRISISWEAVE LOCATION VERIFICATION")
print("======================================")

print(
    "Status:",
    location_verification["status"]
)

print(
    "GIS Score:",
    location_verification["verification_score"],
    "/ 5"
)

print()

print(
    "Water relationship:",
    location_verification["water_relevant"]
)

print(
    "Water distance:",
    location_verification["water_distance_m"],
    "meters"
)

print(
    "Road relationship:",
    location_verification["road_relevant"]
)

print(
    "Road distance:",
    location_verification["road_distance_m"],
    "meters"
)

CRISISWEAVE LOCATION VERIFICATION
Status: GEOGRAPHICALLY PLAUSIBLE
GIS Score: 5 / 5

Water relationship: True
Water distance: 73.52 meters
Road relationship: True
Road distance: 17.2 meters


GIS-25 — Build Candidate Locations

In [29]:
# ==========================================
# GIS-25: CANDIDATE LOCATIONS
# ==========================================

candidate_locations = []

# ------------------------------------------
# Schools
# ------------------------------------------

for idx, row in schools.iterrows():

    geom = row.geometry

    if geom.geom_type == "Point":
        point = geom
    else:
        point = geom.centroid

    name = row.get("name", "School")

    if pd.isna(name):
        name = "School"

    candidate_locations.append({
        "candidate_id": f"SCHOOL_{idx}",
        "type": "School",
        "name": str(name),
        "latitude": float(point.y),
        "longitude": float(point.x)
    })


# ------------------------------------------
# Hospitals
# ------------------------------------------

for idx, row in hospitals.iterrows():

    geom = row.geometry

    if geom.geom_type == "Point":
        point = geom
    else:
        point = geom.centroid

    name = row.get("name", "Hospital")

    if pd.isna(name):
        name = "Hospital"

    candidate_locations.append({
        "candidate_id": f"HOSPITAL_{idx}",
        "type": "Hospital",
        "name": str(name),
        "latitude": float(point.y),
        "longitude": float(point.x)
    })


# ------------------------------------------
# Bridges
# ------------------------------------------

for idx, row in bridges.iterrows():

    geom = row.geometry

    point = geom.representative_point()

    name = row.get("name", "Bridge")

    if pd.isna(name):
        name = "Bridge"

    candidate_locations.append({
        "candidate_id": f"BRIDGE_{idx}",
        "type": "Bridge",
        "name": str(name),
        "latitude": float(point.y),
        "longitude": float(point.x)
    })


candidate_df = pd.DataFrame(
    candidate_locations
)

print("Candidate locations created:", len(candidate_df))

candidate_df

Candidate locations created: 4


,candidate_id,type,name,latitude,longitude
0,SCHOOL_0,School,Kendriya Vidyalaya-Mundamveli Kochi,9.930821,76.267321
1,HOSPITAL_0,Hospital,AMJ Hospital,9.929430,76.270290
2,BRIDGE_0,Bridge,Gateway of Kochi Bridge,9.933706,76.269885
3,BRIDGE_1,Bridge,Bridge,9.930450,76.266954


GIS-26 — Calculate distance from incident

In [30]:
# ==========================================
# GIS-26: CANDIDATE DISTANCES
# ==========================================

candidate_points = gpd.GeoDataFrame(
    candidate_df.copy(),
    geometry=[
        Point(lon, lat)
        for lat, lon in zip(
            candidate_df["latitude"],
            candidate_df["longitude"]
        )
    ],
    crs="EPSG:4326"
)

candidate_metric = candidate_points.to_crs(
    METRIC_CRS
)

incident_geom = incident_metric.geometry.iloc[0]

candidate_metric["distance_m"] = (
    candidate_metric.geometry
    .distance(incident_geom)
)

candidate_metric[
    "distance_m"
] = candidate_metric[
    "distance_m"
].round(2)

candidate_df = pd.DataFrame(
    candidate_metric.drop(
        columns="geometry"
    )
)

candidate_df = candidate_df.sort_values(
    "distance_m"
).reset_index(drop=True)

candidate_df

,candidate_id,type,name,latitude,longitude,distance_m
0,HOSPITAL_0,Hospital,AMJ Hospital,9.929430,76.270290,38.14
1,SCHOOL_0,School,Kendriya Vidyalaya-Mundamveli Kochi,9.930821,76.267321,390.06
2,BRIDGE_1,Bridge,Bridge,9.930450,76.266954,409.23
3,BRIDGE_0,Bridge,Gateway of Kochi Bridge,9.933706,76.269885,511.08


GIS-27 — Add basic priority type

In [31]:
# ==========================================
# GIS-27: BASE CRITICALITY
# ==========================================

criticality_scores = {
    "Hospital": 1.00,
    "Bridge": 0.90,
    "School": 0.75
}

candidate_df["base_criticality"] = (
    candidate_df["type"]
    .map(criticality_scores)
    .fillna(0.50)
)

candidate_df

,candidate_id,type,name,latitude,longitude,distance_m,base_criticality
0,HOSPITAL_0,Hospital,AMJ Hospital,9.929430,76.270290,38.14,1.00
1,SCHOOL_0,School,Kendriya Vidyalaya-Mundamveli Kochi,9.930821,76.267321,390.06,0.75
2,BRIDGE_1,Bridge,Bridge,9.930450,76.266954,409.23,0.90
3,BRIDGE_0,Bridge,Gateway of Kochi Bridge,9.933706,76.269885,511.08,0.90


GIS-28 — Create the candidate layer on the map

In [32]:
# ==========================================
# GIS-28: CANDIDATE LOCATIONS ON MAP
# ==========================================

candidate_layer = folium.FeatureGroup(
    name="Candidate Locations"
)

for _, row in candidate_df.iterrows():

    if row["type"] == "School":
        icon_name = "graduation-cap"

    elif row["type"] == "Hospital":
        icon_name = "plus"

    elif row["type"] == "Bridge":
        icon_name = "road"

    else:
        icon_name = "info-sign"

    folium.Marker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        tooltip=(
            f"{row['type']}: "
            f"{row['name']}"
        ),
        popup=f"""
        <b>Candidate Location</b><br>
        Type: {row['type']}<br>
        Name: {row['name']}<br>
        Distance from incident:
        {row['distance_m']:.1f} m
        """,
        icon=folium.Icon(
            icon=icon_name,
            prefix="fa"
        )
    ).add_to(candidate_layer)


candidate_layer.add_to(crisis_map)

folium.LayerControl().add_to(crisis_map)

crisis_map

GIS-29 — Candidate Spatial Relationships

In [33]:
# ==========================================
# GIS-29: CANDIDATE SPATIAL RELATIONSHIPS
# ==========================================

def calculate_candidate_relationships(
    candidate_df,
    incident_metric,
    water_metric,
    roads_metric
):

    results = []

    incident_geom = incident_metric.geometry.iloc[0]

    for _, candidate in candidate_df.iterrows():

        candidate_point = Point(
            candidate["longitude"],
            candidate["latitude"]
        )

        candidate_gdf = gpd.GeoDataFrame(
            geometry=[candidate_point],
            crs="EPSG:4326"
        )

        candidate_metric_point = (
            candidate_gdf
            .to_crs(METRIC_CRS)
            .geometry.iloc[0]
        )

        # ----------------------------------
        # Distance from incident
        # ----------------------------------

        incident_distance = (
            candidate_metric_point
            .distance(incident_geom)
        )

        # ----------------------------------
        # Distance to waterbody
        # ----------------------------------

        if len(water_metric) > 0:

            water_distances = (
                water_metric.geometry
                .distance(candidate_metric_point)
            )

            water_distance = float(
                water_distances.min()
            )

        else:

            water_distance = None

        # ----------------------------------
        # Distance to road
        # ----------------------------------

        if len(roads_metric) > 0:

            road_distances = (
                roads_metric.geometry
                .distance(candidate_metric_point)
            )

            road_distance = float(
                road_distances.min()
            )

        else:

            road_distance = None

        results.append({
            "candidate_id":
                candidate["candidate_id"],

            "type":
                candidate["type"],

            "name":
                candidate["name"],

            "latitude":
                candidate["latitude"],

            "longitude":
                candidate["longitude"],

            "distance_from_incident_m":
                round(incident_distance, 2),

            "distance_to_water_m":
                round(water_distance, 2)
                if water_distance is not None
                else None,

            "distance_to_road_m":
                round(road_distance, 2)
                if road_distance is not None
                else None,

            "base_criticality":
                candidate["base_criticality"]
        })

    return pd.DataFrame(results)


candidate_relationships = (
    calculate_candidate_relationships(
        candidate_df,
        incident_metric,
        water_metric,
        roads_metric
    )
)

candidate_relationships

,candidate_id,type,name,latitude,longitude,distance_from_incident_m,distance_to_water_m,distance_to_road_m,base_criticality
0,HOSPITAL_0,Hospital,AMJ Hospital,9.929430,76.270290,38.14,57.29,19.62,1.00
1,SCHOOL_0,School,Kendriya Vidyalaya-Mundamveli Kochi,9.930821,76.267321,390.06,33.99,28.47,0.75
2,BRIDGE_1,Bridge,Bridge,9.930450,76.266954,409.23,9.14,0.00,0.90
3,BRIDGE_0,Bridge,Gateway of Kochi Bridge,9.933706,76.269885,511.08,0.00,0.00,0.90


GIS-30 — Classify spatial proximity

In [34]:
# ==========================================
# GIS-30: SPATIAL PROXIMITY CLASSIFICATION
# ==========================================

def classify_proximity(distance):

    if distance is None:
        return "Unknown"

    if distance <= 100:
        return "Very Close"

    elif distance <= 250:
        return "Close"

    elif distance <= 500:
        return "Moderate"

    else:
        return "Far"


candidate_relationships[
    "incident_proximity"
] = candidate_relationships[
    "distance_from_incident_m"
].apply(classify_proximity)


candidate_relationships[
    "water_proximity"
] = candidate_relationships[
    "distance_to_water_m"
].apply(classify_proximity)


candidate_relationships[
    "road_proximity"
] = candidate_relationships[
    "distance_to_road_m"
].apply(classify_proximity
)


candidate_relationships

,candidate_id,type,name,latitude,longitude,distance_from_incident_m,distance_to_water_m,distance_to_road_m,base_criticality,incident_proximity,water_proximity,road_proximity
0,HOSPITAL_0,Hospital,AMJ Hospital,9.929430,76.270290,38.14,57.29,19.62,1.00,Very Close,Very Close,Very Close
1,SCHOOL_0,School,Kendriya Vidyalaya-Mundamveli Kochi,9.930821,76.267321,390.06,33.99,28.47,0.75,Moderate,Very Close,Very Close
2,BRIDGE_1,Bridge,Bridge,9.930450,76.266954,409.23,9.14,0.00,0.90,Moderate,Very Close,Very Close
3,BRIDGE_0,Bridge,Gateway of Kochi Bridge,9.933706,76.269885,511.08,0.00,0.00,0.90,Far,Very Close,Very Close


GIS-31 — Create a preliminary spatial priority

In [35]:
# ==========================================
# GIS-31: PRELIMINARY SPATIAL PRIORITY
# ==========================================

def proximity_score(distance):

    if distance is None:
        return 0.0

    if distance <= 100:
        return 1.0

    elif distance <= 250:
        return 0.8

    elif distance <= 500:
        return 0.5

    else:
        return 0.2


candidate_relationships[
    "incident_proximity_score"
] = candidate_relationships[
    "distance_from_incident_m"
].apply(proximity_score)


candidate_relationships[
    "water_proximity_score"
] = candidate_relationships[
    "distance_to_water_m"
].apply(proximity_score)


candidate_relationships[
    "spatial_priority"
] = (
    0.50 *
    candidate_relationships[
        "incident_proximity_score"
    ]
    +
    0.30 *
    candidate_relationships[
        "water_proximity_score"
    ]
    +
    0.20 *
    candidate_relationships[
        "base_criticality"
    ]
)


candidate_relationships[
    "spatial_priority"
] = candidate_relationships[
    "spatial_priority"
].round(3)


candidate_relationships = (
    candidate_relationships
    .sort_values(
        "spatial_priority",
        ascending=False
    )
    .reset_index(drop=True)
)

candidate_relationships

,candidate_id,type,name,latitude,longitude,distance_from_incident_m,distance_to_water_m,distance_to_road_m,base_criticality,incident_proximity,water_proximity,road_proximity,incident_proximity_score,water_proximity_score,spatial_priority
0,HOSPITAL_0,Hospital,AMJ Hospital,9.929430,76.270290,38.14,57.29,19.62,1.00,Very Close,Very Close,Very Close,1.0,1.0,1.00
1,BRIDGE_1,Bridge,Bridge,9.930450,76.266954,409.23,9.14,0.00,0.90,Moderate,Very Close,Very Close,0.5,1.0,0.73
2,SCHOOL_0,School,Kendriya Vidyalaya-Mundamveli Kochi,9.930821,76.267321,390.06,33.99,28.47,0.75,Moderate,Very Close,Very Close,0.5,1.0,0.70
3,BRIDGE_0,Bridge,Gateway of Kochi Bridge,9.933706,76.269885,511.08,0.00,0.00,0.90,Far,Very Close,Very Close,0.2,1.0,0.58


GIS-32 — Save final GIS output

In [36]:
import os
import json
import pandas as pd

GIS_RESULTS_DIR = "/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/GIS"
os.makedirs(GIS_RESULTS_DIR, exist_ok=True)

# Final GIS package
gis_output = {
    "incident_id": crisis_incident["incident_id"],

    "incident_location": crisis_incident["location"],

    "gis_context": crisis_incident["gis_context"],

    "location_verification": location_verification,

    "candidate_locations": candidate_relationships.to_dict(orient="records")
}

# Save complete GIS result
gis_json_path = os.path.join(
    GIS_RESULTS_DIR,
    "CrisisWeave_GIS_CW001.json"
)

with open(gis_json_path, "w", encoding="utf-8") as f:
    json.dump(gis_output, f, indent=2, ensure_ascii=False, default=str)

# Save candidate locations separately for later recommendation module
candidate_csv_path = os.path.join(
    GIS_RESULTS_DIR,
    "CrisisWeave_GIS_Candidate_Locations_CW001.csv"
)

candidate_relationships.to_csv(candidate_csv_path, index=False)

print("GIS output saved successfully.")
print("JSON:", gis_json_path)
print("CSV :", candidate_csv_path)

GIS output saved successfully.
JSON: /content/gdrive/MyDrive/CrisisWeave_Project/04_Results/GIS/CrisisWeave_GIS_CW001.json
CSV : /content/gdrive/MyDrive/CrisisWeave_Project/04_Results/GIS/CrisisWeave_GIS_Candidate_Locations_CW001.csv


In [37]:
print("\nGIS OUTPUT SUMMARY")
print("------------------")
print("Incident ID:", gis_output["incident_id"])
print("Location:", gis_output["incident_location"])
print("Verification:", gis_output["location_verification"]["status"])
print("Candidates:", len(gis_output["candidate_locations"]))


GIS OUTPUT SUMMARY
------------------
Incident ID: CW-001
Location: {'latitude': 9.929117, 'longitude': 76.270436, 'source': 'user_map_pin'}
Verification: GEOGRAPHICALLY PLAUSIBLE
Candidates: 4
